# Ziqra.ai — Remote GPU Training (Google Colab)

This notebook is **infrastructure only**. All training logic — model loading, LoRA, the dataset pipeline, the terminal UI — lives in `training/trainer/` in the project repo. This notebook installs dependencies, fetches the project onto the Colab GPU, and runs the exact same command you'd run locally:

```
python -m backend.knowledge_distillation.training.trainer.train --config backend/knowledge_distillation/training/configs/<coach>.yaml
```

**Before running:** `backend/knowledge_distillation/` (the trainer, configs, prompts, and your prepared dataset) must be committed and pushed to your GitHub repo. Gemma additionally requires a Hugging Face account with its gated license accepted (Cell 2) — Qwen does not.

## Cell 1 — Install dependencies

In [ ]:
# torch already ships with CUDA support on Colab's GPU runtime; everything else
# the trainer needs is installed here. No training logic in this cell.
!pip install -q transformers peft accelerate bitsandbytes rich pyyaml


## Cell 2 — Authenticate to Hugging Face (required for gated models like Gemma)

In [ ]:
# This Colab session is a separate, ephemeral machine -- any earlier
# huggingface-cli login / huggingface_hub.login() on your own computer does not
# carry over here. Store your token as a Colab secret instead of pasting it into
# a cell: click the key icon in the left sidebar, add a secret named HF_TOKEN,
# and toggle "Notebook access" on for this notebook.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

# Alternative if you don't want to use Colab Secrets -- pops up an interactive
# widget to paste your token into instead:
# from huggingface_hub import notebook_login
# notebook_login()


## Cell 3 — Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/tahapathan2603/Ziqra-ai-integration.git"
PROJECT_DIR = "/content/Ziqra-ai-integration"

# Pull on every run, not just clone-if-missing -- otherwise a Colab session that
# already has PROJECT_DIR from an earlier run silently keeps using a stale
# checkout and never picks up new commits (e.g. trainer fixes).
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull origin main

%cd {PROJECT_DIR}


## Cell 4 — Mount Google Drive (if required)

In [ ]:
# Used for two things only: (a) pulling in the prepared dataset if you chose not
# to commit it to git, and (b) as the durable destination for checkpoints/adapters
# at the end of the run, since Colab's local disk is wiped when the runtime
# disconnects. Skip this cell entirely if your dataset is already in the cloned
# repo and you don't need the outputs to survive a disconnect -- Cells 7/8 detect
# whether Drive is mounted and degrade gracefully if it isn't.
from google.colab import drive

drive.mount("/content/drive")

# Optional: uncomment if your dataset lives on Drive instead of in git.
# !mkdir -p backend/knowledge_distillation/training/data/datasets
# !cp -r /content/drive/MyDrive/ziqra_dataset/* backend/knowledge_distillation/training/data/datasets/


## Cell 5 — Configure paths

In [ ]:
# The only environment-specific "configuration" this notebook does: which
# coach's YAML to train, and confirming a GPU is attached. Nothing about the
# training run itself is configured here -- that all lives in
# training/configs/*.yaml, read entirely by the trainer.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

CONFIG_PATH = "backend/knowledge_distillation/training/configs/articulation.yaml"
# CONFIG_PATH = "backend/knowledge_distillation/training/configs/delivery.yaml"

print("Training with config:", CONFIG_PATH)


## Cell 6 — Run the trainer

In [ ]:
# This is the entire training step -- identical command whether it runs on
# Colab, RunPod, Kaggle, Lambda Labs, or your own machine. The Rich terminal UI
# (progress bar, live loss/LR/GPU-memory panel) renders directly in this cell's
# output.
#
# train.py uses relative imports (it is a package module, not a standalone
# script), and its own path is backend/knowledge_distillation/training/trainer/
# -- both facts mean it must be run with -m from the project root, not as a
# plain script path.
!python -m backend.knowledge_distillation.training.trainer.train --config {CONFIG_PATH}


## Cell 7 — Save checkpoints

In [ ]:
# Intermediate checkpoints (optimizer state, periodic saves) -- large, and only
# useful if you want to resume training later. Skip if you only care about the
# final adapter (next cell). Safe to run even if Cell 4 (Drive mount) was
# skipped -- it just prints a note instead of copying anywhere.
import shutil
from pathlib import Path

import yaml

config = yaml.safe_load(open(CONFIG_PATH))
output_dir = Path(config["checkpointing"]["output_dir"])
checkpoints_dir = output_dir / "checkpoints"
drive_mounted = Path("/content/drive/MyDrive").exists()

if not checkpoints_dir.exists():
    print(f"No checkpoints found at {checkpoints_dir}")
elif drive_mounted:
    dest = Path("/content/drive/MyDrive/ziqra_training_outputs") / output_dir.name / "checkpoints"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(checkpoints_dir, dest, dirs_exist_ok=True)
    print(f"Checkpoints copied to {dest}")
else:
    print("Drive not mounted (Cell 4 skipped) -- checkpoints remain local at", checkpoints_dir)


## Cell 8 — Export the final LoRA adapter

In [ ]:
# The one artifact you actually need afterward: the small, final adapter, stored
# separately from the larger intermediate checkpoints (see Cell 7). Copied to
# Drive if mounted, and always zipped for a direct browser download.
import shutil
from pathlib import Path

import yaml
from google.colab import files

config = yaml.safe_load(open(CONFIG_PATH))
output_dir = Path(config["checkpointing"]["output_dir"])
adapters_dir = output_dir / "adapters"

if not adapters_dir.exists():
    raise FileNotFoundError(f"No adapter found at {adapters_dir} -- did training in Cell 6 complete?")

if Path("/content/drive/MyDrive").exists():
    dest = Path("/content/drive/MyDrive/ziqra_training_outputs") / output_dir.name / "adapters"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(adapters_dir, dest, dirs_exist_ok=True)
    print(f"Adapter copied to {dest}")

zip_path = shutil.make_archive(f"/content/{output_dir.name}_adapter", "zip", adapters_dir)
print(f"Adapter zipped: {zip_path}")
files.download(zip_path)
